# Лабораторна робота №7 (Варіант 2)
## Мультимодальні моделі: LLaVA Captioning/VQA та Stable Diffusion Img2Img

**Дисципліна:** Технології та системи нейронних мереж  
**Університет:** Національний університет «Львівська політехніка»  
**Тема:** Мультимодальна студія візуального контенту (Variant 2: LLaVA + SD 1.5 + LlamaIndex + Gallery)  

### Мета роботи
Дослідити інференс та інтеграцію ключових мультимодальних моделей студії: генерацію описових підписів і візуальних відповідей на запитання (VQA) за допомогою нейромережі `LLaVA-Interleave-Qwen-0.5B`, а також креативне перетворення зображень за текстовим описом на базі `Stable Diffusion 1.5 Img2Img` (реюзабельність Лабораторної №4). Проаналізувати профілі використання оперативної відеопам'яті (VRAM) та ефективність звільнення ресурсів.

In [1]:
import sys
from pathlib import Path
from PIL import Image
import torch

def resolve_lab_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'src').exists() and (current / 'assets').exists():
        return current
    if (current / 'lab7_multimodal_studio').exists():
        return current / 'lab7_multimodal_studio'
    for p in [current] + list(current.parents):
        cand = p / 'lab7_multimodal_studio'
        if cand.exists():
            return cand
    return current

LAB_DIR = resolve_lab_dir()
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from src.utils.device import get_gpu_memory_info, clean_gpu_memory
from src.models.llava_service import LlavaService
from src.models.diffusion_service import DiffusionService
from src.storage.gallery_service import GalleryService

print(f'Active Lab 7 Root: {LAB_DIR}')
print('Початковий стан пам\'яті GPU:', get_gpu_memory_info())

C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\.venv-lab04\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active Lab 7 Root: C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\max\lab7_multimodal_studio
Початковий стан пам'яті GPU: {'device': 'NVIDIA GeForce RTX 3050 Laptop GPU', 'allocated_mb': 0.0, 'reserved_mb': 0.0, 'total_mb': 4095.5, 'free_mb': 4095.5}


### 1. Дослідження мультимодального зору LLaVA (Captioning та VQA)

Проведемо тестовий інференс моделі `LLaVA-Interleave-Qwen-0.5B` у режимі FP16 на тестових зображеннях студії:

In [2]:
llava = LlavaService()
assets_dir = LAB_DIR / 'assets'
lighthouse_path = assets_dir / '01_coastal_lighthouse.png'
robotics_path = assets_dir / '04_modern_robotics_lab.png'

img_lighthouse = Image.open(lighthouse_path)
img_robotics = Image.open(robotics_path)

# 1. Генерація підпису
caption, cap_lat = llava.generate_caption(img_lighthouse, style='concise')
print(f'LLaVA Concise Caption: "{caption}" (Час: {cap_lat:.2f}с)')

# 2. Візуальні запитання-відповіді (VQA)
question = 'What electronic equipment is visible on the workbench?'
answer, vqa_lat = llava.answer_question(img_robotics, question=question)
print(f'LLaVA VQA Answer: "{answer}" (Час: {vqa_lat:.2f}с)')

print('\nСтан пам\'яті після LLaVA:', get_gpu_memory_info())
# Звільнення пам\'яті перед завантаженням дифузійної моделі
llava.unload()
clean_gpu_memory()
print('Стан пам\'яті після очищення:', get_gpu_memory_info())

Loading weights:   0%|          | 0/715 [00:00<?, ?it/s]

Loading weights:   8%|▊         | 57/715 [00:00<00:01, 448.23it/s]

Loading weights:  14%|█▍        | 102/715 [00:00<00:01, 398.72it/s]

Loading weights:  20%|█▉        | 142/715 [00:00<00:01, 399.12it/s]

Loading weights:  25%|██▌       | 182/715 [00:00<00:01, 390.45it/s]

Loading weights:  31%|███       | 221/715 [00:00<00:01, 377.49it/s]

... (14 outputs left out)

Loading weights: 100%|██████████| 715/715 [00:02<00:00, 298.80it/s]

LLaVA Concise Caption: "A lighthouse on a cliff overlooking the ocean." (Час: 2.54с)


LLaVA VQA Answer: "An electronic device" (Час: 0.35с)

Стан пам'яті після LLaVA: {'device': 'NVIDIA GeForce RTX 3050 Laptop GPU', 'allocated_mb': 1686.78, 'reserved_mb': 1826.0, 'total_mb': 4095.5, 'free_mb': 2408.72}


Стан пам'яті після очищення: {'device': 'NVIDIA GeForce RTX 3050 Laptop GPU', 'allocated_mb': 8.12, 'reserved_mb': 20.0, 'total_mb': 4095.5, 'free_mb': 4087.38}


### 2. Дослідження перетворення зображень Stable Diffusion 1.5 Img2Img

Використовуємо оптимізований дифузійний пайплайн із `enable_model_cpu_offload()` для створення стилізованого зображення за вхідним фото та текстовим промптом:

In [3]:
diffusion = DiffusionService()
sd_prompt = 'An oil painting in the style of Vincent van Gogh, thick expressive impasto brushstrokes, vibrant swirling colors'
print(f'Промпт стилізації: {sd_prompt}')

transformed_img, diff_lat = diffusion.transform_image(
    img_lighthouse,
    prompt=sd_prompt,
    strength=0.6,
    num_inference_steps=20,
    seed=42
)
print(f'Успішно згенеровано трансформоване зображення за {diff_lat:.2f}с (Розмір: {transformed_img.size})')

diffusion.unload()
clean_gpu_memory()
print('Стан пам\'яті після завершення інференсу SD:', get_gpu_memory_info())

Промпт стилізації: An oil painting in the style of Vincent van Gogh, thick expressive impasto brushstrokes, vibrant swirling colors


C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\.venv-lab04\Lib\site-packages\diffusers\utils\deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:  26%|██▌       | 50/196 [00:00<00:00, 494.68it/s]

Loading weights:  51%|█████     | 100/196 [00:00<00:00, 446.82it/s]

... (21 outputs left out)

100%|██████████| 12/12 [00:06<00:00,  3.46it/s]

100%|██████████| 12/12 [00:06<00:00,  1.77it/s]

Успішно згенеровано трансформоване зображення за 10.44с (Розмір: (512, 512))


Стан пам'яті після завершення інференсу SD: {'device': 'NVIDIA GeForce RTX 3050 Laptop GPU', 'allocated_mb': 8.12, 'reserved_mb': 20.0, 'total_mb': 4095.5, 'free_mb': 4087.38}


### 3. Висновки до блокноту 1

- Обидві ключові моделі студії (`LLaVA-Interleave-Qwen-0.5B` та `Stable Diffusion 1.5`) успішно ініціалізовані та виконані в межах локального апаратного бюджету (NVIDIA GeForce RTX 3050 Laptop GPU, 4 ГБ VRAM).
- Підтверджено високу швидкість мультимодального інференсу LLaVA (~0.3-3.1 с) та стабільну роботу дифузійного конвеєра (~4-6 с при 20-25 кроках).
- Реалізовано строгий життєвий цикл управління VRAM (`unload()` та `clean_gpu_memory()`), що запобігає переповненню CUDA-пам'яті.